# Name That LLM · Open Pool CPU CV

Attach the competition, Open Pool v19 and ntl-cv-assets. Select CPU and Internet OFF. Run all cells once in a fresh session. Download metrics.json and the validated submission.

In [ ]:
from pathlib import Path
import json
import math
import os
import time
import warnings
from concurrent.futures import as_completed
from joblib.externals.loky import ProcessPoolExecutor
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import balanced_accuracy_score
from threadpoolctl import threadpool_limits

START = time.monotonic()
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
EXPECTED_FOLDS = 'e8eb76023a9679528b5f471ab0cdb6f00e7292b177ce5cea69fc2d8a15393106'
EXPECTED_POOL = 'fe6ece20e6f9f2fe5fb3c47f2aaa9749394c1056bc688a8eab865e0dc2bb56df'
SMALL, LARGE = 1100, 2500
K0_TARGET, K0_TOLERANCE = 0.43267, 0.005
MIN_GAIN = 0.005
BASELINE = 0.33929903
CONFIG = {'seed': 42, 'validation_fraction': 0.2, 'tfidf': {'analyzer': 'char', 'ngram_range': [3, 5], 'lowercase': False, 'min_df': 2, 'max_features': 40000, 'sublinear_tf': True}, 'word_tfidf': {'analyzer': 'word', 'ngram_range': [1, 1], 'lowercase': False, 'min_df': 2, 'max_features': 40000, 'sublinear_tf': True, 'token_pattern': '(?u)\\b\\w+\\b|[^\\w\\s]'}, 'logistic_regression': {'C': 1.0, 'solver': 'lbfgs', 'max_iter': 400, 'tol': 1e-06, 'class_weight': 'balanced'}}


In [ ]:
baseline_source = r'''"""CPU TF-IDF + Logistic Regression. Run: python -m src.baseline --help.

No network calls. Only load joblib artifacts you created/trust: joblib uses pickle.
"""
import argparse
import csv
import hashlib
import importlib.metadata
import json
import platform
import time
import unicodedata
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from threadpoolctl import threadpool_limits

# Official Overview > Evaluation > Valid Labels, verified 2026-10-26.
# This display order is NOT an order of probability columns in the submission.
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra',
          'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash',
          'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5',
          'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']


def require(condition, message):
    if not condition:
        raise ValueError(message)


def sha256(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('x', encoding='utf-8') as f:
        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)


def save_csv(path, frame):
    with Path(path).open('x', encoding='utf-8', newline='') as f:
        frame.to_csv(f, index=False)


def environment():
    return {'python': platform.python_version(), 'platform': platform.platform(),
            'libraries': {name: importlib.metadata.version(name) for name in
                          ['numpy', 'scipy', 'pandas', 'scikit-learn', 'pyarrow',
                           'joblib', 'threadpoolctl']}}


def normalized(text):
    # Used only for duplicate grouping, not as model input.
    return ' '.join(unicodedata.normalize('NFC', text).split())


def read_input(path, labeled=False):
    path = Path(path)
    require(path.suffix.lower() == '.csv', 'Input must be a CSV file.')
    with path.open(encoding='utf-8-sig', newline='') as f:
        header = next(csv.reader(f), [])
    require(len(header) == len(set(header)), 'Duplicate column headers are not allowed.')
    frame = pd.read_csv(path, dtype='string', keep_default_na=False,
                        na_values=[''], skip_blank_lines=False)
    required = {'id', 'text'} | ({'model'} if labeled else set())
    require(required <= set(frame), f'Missing columns: {sorted(required - set(frame))}')
    require(not frame['id'].isna().any(), 'IDs contain null values.')
    require(not frame['id'].str.strip().eq('').any(), 'IDs contain blank values.')
    require(not frame['id'].duplicated().any(), 'IDs must be unique.')
    null_count = int(frame['text'].isna().sum())
    frame['text'] = frame['text'].fillna('')
    blank_count = int(frame['text'].str.strip().eq('').sum())
    audit = {'rows': len(frame), 'columns': list(frame), 'null_text': null_count,
             'blank_text_including_null': blank_count,
             'sha256': sha256(path), 'bytes': path.stat().st_size}
    if labeled:
        require(len(frame) > 0, 'Training data is empty.')
        require(blank_count == 0, 'Blank/null training texts are rejected; fix the data source.')
        require(not frame['model'].isna().any(), 'Training labels contain null values.')
        unknown = sorted(set(frame['model']) - set(LABELS))
        missing = sorted(set(LABELS) - set(frame['model']))
        require(not unknown, f'Unknown model labels: {unknown}')
        require(not missing, f'Missing official training classes: {missing}; no synthetic classes added.')
        if 'prompt_id' in frame:
            require(not frame['prompt_id'].isna().any() and
                    not frame['prompt_id'].str.strip().eq('').any(), 'prompt_id contains null/blank values.')
        audit['class_counts'] = {str(k): int(v) for k, v in frame['model'].value_counts().items()}
    elif blank_count:
        warnings.warn(f'{blank_count} blank/null test texts kept as empty strings; no rows dropped.')
    return frame, audit


def connected_groups(frame):
    """Connected components combine prompt groups AND duplicate-text groups.

    A duplicate connecting two prompts must keep both entire prompts together.
    Without prompt_id, only duplicate grouping is available, a weaker guarantee.
    """
    parent = list(range(len(frame)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        parent[find(b)] = find(a)

    keys = frame['text'].map(normalized)
    for values in ([frame['prompt_id'].tolist()] if 'prompt_id' in frame else []) + [keys.tolist()]:
        seen = {}
        for i, value in enumerate(values):
            if value in seen:
                union(i, seen[value])
            else:
                seen[value] = i
    groups = np.array([find(i) for i in range(len(frame))])
    return groups, keys


def check_disjoint(frame, groups, train_idx, valid_idx):
    """Stop on missing classes or any group/text/prompt overlap between the two sides."""
    a, b = frame.iloc[train_idx], frame.iloc[valid_idx]
    absent = {name: sorted(set(LABELS) - set(part['model'])) for name, part in
              [('train', a), ('validation', b)]}
    require(not any(absent.values()), f'Holdout lacks classes: {absent}; evaluation aborted.')
    require(not (set(groups[train_idx]) & set(groups[valid_idx])), 'Group leakage detected.')
    exact_overlap = len(set(a['text']) & set(b['text']))
    normalized_overlap = len(set(a['text'].map(normalized)) & set(b['text'].map(normalized)))
    prompt_overlap = (len(set(a['prompt_id']) & set(b['prompt_id'])) if 'prompt_id' in frame else None)
    require(exact_overlap == normalized_overlap == 0 and prompt_overlap in (0, None), 'Text/prompt leakage detected.')
    return absent, exact_overlap, normalized_overlap, prompt_overlap


def make_split(frame, config):
    """One grouped holdout over connected prompt/duplicate components."""
    groups, keys = connected_groups(frame)
    require(len(set(groups)) >= 2, 'Not enough independent groups for a holdout.')
    splitter = GroupShuffleSplit(n_splits=1, test_size=config['validation_fraction'],
                                random_state=config['seed'])
    train_idx, valid_idx = next(splitter.split(frame['text'], frame['model'], groups))
    a, b = frame.iloc[train_idx], frame.iloc[valid_idx]
    absent, exact_overlap, normalized_overlap, prompt_overlap = check_disjoint(frame, groups, train_idx, valid_idx)
    conflicts = frame.assign(key=keys).groupby('key')['model'].nunique()
    audit = {'method': 'GroupShuffleSplit on connected prompt/duplicate components',
             'seed': config['seed'], 'validation_fraction': config['validation_fraction'],
             'group_metadata': 'prompt_id' if 'prompt_id' in frame else 'duplicate text only; prompt/source unavailable',
             'component_count': len(set(groups)), 'train_rows': len(a), 'validation_rows': len(b),
             'train_groups': len(set(groups[train_idx])), 'validation_groups': len(set(groups[valid_idx])),
             'prompt_overlap': prompt_overlap, 'exact_text_overlap': exact_overlap,
             'normalized_text_overlap': normalized_overlap,
             'exact_duplicate_extra_rows': int(frame['text'].duplicated().sum()),
             'normalized_duplicate_extra_rows': int(keys.duplicated().sum()),
             'normalized_duplicate_conflicting_label_groups': int((conflicts > 1).sum()),
             'absent_classes': absent,
             'class_counts': {name: {label: int((part['model'] == label).sum()) for label in LABELS}
                              for name, part in [('train', a), ('validation', b)]}}
    return train_idx, valid_idx, groups, audit


def _tfidf(params):
    params = dict(params)
    params['ngram_range'] = tuple(params['ngram_range'])  # JSON lists are rejected by sklearn
    return TfidfVectorizer(**params, dtype=np.float32)  # float32 on every branch, or hstack upcasts


def build_pipeline(config):
    features = _tfidf(config['tfidf'])
    if 'word_tfidf' in config:  # optional word branch (debates/2-them-word-tfidf.md)
        features = FeatureUnion([('char', features), ('word', _tfidf(config['word_tfidf']))])
    return Pipeline([('tfidf', features),
                     ('classifier', LogisticRegression(**config['logistic_regression'],
                                                       random_state=config['seed']))])


def feature_counts(model):
    step = model.named_steps['tfidf']
    branches = step.transformer_list if isinstance(step, FeatureUnion) else [('char', step)]
    return {name: len(vectorizer.vocabulary_) for name, vectorizer in branches}


def fit_checked(model, texts, labels):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(texts, labels)
    convergence = [str(w.message) for w in caught if issubclass(w.category, ConvergenceWarning)]
    require(not convergence, f'Logistic Regression did not converge: {convergence}')
    classifier = model.named_steps['classifier']
    require(np.any(classifier.n_iter_ > 0) and np.any(classifier.coef_ != 0),
            'Optimizer stopped without learning coefficients; inspect numerical tolerance.')
    return time.perf_counter() - start


def score(labels, predictions):
    return {'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),
            'accuracy': float(accuracy_score(labels, predictions)),
            'macro_f1': float(f1_score(labels, predictions, labels=LABELS, average='macro', zero_division=0))}


def _assign_within(log_p, classes, labels):
    """Each cluster: Hungarian on -log P, class columns repeated ceil(size/K) times."""
    classes = np.asarray(classes)
    result = np.empty(len(log_p), dtype=object)
    for label in np.unique(labels):
        idx = np.flatnonzero(labels == label)
        repeats = -(-len(idx) // len(classes))
        rows, cols = linear_sum_assignment(-np.tile(log_p[idx], (1, repeats)))
        result[idx[rows]] = classes[cols % len(classes)]
    return result


def assign_by_topic(texts, probabilities, classes):
    """Optional post-processing (debates/1-huong-cai-tien-mo-hinh.md). Off by default.

    Clusters rows by word topic (k = round(m/K)) and gives each cluster distinct labels.
    Falls back to argmax for n < K or an empty vocabulary; zero-vector rows keep argmax.
    Gain shrinks when per-model counts are unequal (debate 1, E9).
    """
    classes = np.asarray(classes)
    result = classes[probabilities.argmax(axis=1)] if len(texts) else np.array([], dtype=object)
    if len(texts) < len(classes):
        return result
    try:
        x = TfidfVectorizer(analyzer='word', stop_words='english', min_df=2,
                            sublinear_tf=True).fit_transform(texts)
    except ValueError:  # empty vocabulary
        return result
    usable = np.asarray(x.getnnz(axis=1) > 0)
    clusters = round(int(usable.sum()) / len(classes))
    if clusters < 1:
        return result
    labels = AgglomerativeClustering(n_clusters=clusters, metric='cosine',
                                     linkage='average').fit_predict(x[usable].toarray())
    result = result.astype(object)
    result[usable] = _assign_within(np.log(np.clip(probabilities[usable], 1e-12, 1)), classes, labels)
    return result


def evaluate_models(frame, train_idx, valid_idx, model, dummy, directory):
    """Evaluate held-out rows only. The full-data inference model is never used here."""
    valid = frame.iloc[valid_idx]
    with threadpool_limits(limits=1):
        pred = model.predict(valid['text'])
    dummy_pred = dummy.predict(np.zeros((len(valid), 1)))
    metrics = {'dummy': score(valid['model'], dummy_pred), 'tfidf_logistic': score(valid['model'], pred)}
    for name, predictions in [('dummy', dummy_pred), ('tfidf_logistic', pred)]:
        report = classification_report(valid['model'], predictions, labels=LABELS,
                                       output_dict=True, zero_division=0)
        save_json(directory / f'{name}_per_class.json', report)
        matrix = pd.DataFrame(confusion_matrix(valid['model'], predictions, labels=LABELS),
                              index=LABELS, columns=LABELS).rename_axis('true_model').reset_index()
        save_csv(directory / f'{name}_confusion_matrix.csv', matrix)
    predictions = valid[['id', 'model']].rename(columns={'model': 'true_model'}).copy()
    predictions['predicted_model'] = pred
    predictions['dummy_model'] = dummy_pred
    save_csv(directory / 'validation_predictions.csv', predictions)
    wrong = valid.loc[np.asarray(valid['model']) != pred, ['id', 'text', 'model']].copy()
    wrong['predicted_model'] = pred[np.asarray(valid['model']) != pred]
    wrong['text'] = wrong['text'].str.slice(0, 180)
    save_json(directory / 'error_examples.json', json.loads(wrong.head(10).to_json(orient='records', force_ascii=False)))
    save_json(directory / 'metrics.json', metrics)
    return metrics


def load_artifact(directory, name):
    directory = Path(directory)
    meta = json.loads((directory / 'metadata.json').read_text(encoding='utf-8'))
    require(meta['environment']['libraries']['scikit-learn'] == importlib.metadata.version('scikit-learn'),
            'scikit-learn version differs from artifact; recreate the pinned environment.')
    require(sha256(directory / name) == meta['artifact_sha256'][name], 'Artifact hash mismatch.')
    # Hash integrity does not make third-party pickle/joblib safe. Trust the producer.
    return joblib.load(directory / name), meta


def train(args):
    start = time.perf_counter()
    config = json.loads(Path(args.config).read_text(encoding='utf-8'))
    frame, input_audit = read_input(args.train, labeled=True)
    train_idx, valid_idx, groups, split_audit = make_split(frame, config)
    artifact, output = Path(args.artifact_dir), Path(args.output_dir)
    require(not artifact.exists() and not output.exists(), 'Use new artifact/output directories; existing runs are never overwritten.')
    artifact.mkdir(parents=True)
    output.mkdir(parents=True)
    print(f'Holdout: {len(train_idx)} train / {len(valid_idx)} validation rows; CPU threads=1', flush=True)
    model = build_pipeline(config)
    fit_seconds = fit_checked(model, frame.iloc[train_idx]['text'], frame.iloc[train_idx]['model'])
    dummy = DummyClassifier(strategy='most_frequent', random_state=config['seed'])
    dummy.fit(np.zeros((len(train_idx), 1)), frame.iloc[train_idx]['model'])
    metrics = evaluate_models(frame, train_idx, valid_idx, model, dummy, output)
    joblib.dump(model, artifact / 'validation_pipeline.joblib', compress=3)
    joblib.dump(dummy, artifact / 'validation_dummy.joblib', compress=3)
    split = frame[['id']].copy()
    split['split'] = 'train'
    split.loc[valid_idx, 'split'] = 'validation'
    split['component'] = groups
    if 'prompt_id' in frame:
        split['prompt_id'] = frame['prompt_id']
    save_csv(artifact / 'split.csv', split)
    save_json(output / 'split_audit.json', split_audit)
    print(json.dumps(metrics), flush=True)
    print('Refitting the same configuration on all labeled rows for inference.', flush=True)
    full = build_pipeline(config)
    full_seconds = fit_checked(full, frame['text'], frame['model'])
    joblib.dump(full, artifact / 'pipeline.joblib', compress=3)
    # Explicitly associate probability columns with the estimator's own class names.
    mapping = {'official_labels': LABELS, 'model_classes': full.classes_.tolist(),
               'probability_column_to_model': {str(i): str(label) for i, label in enumerate(full.classes_)}}
    save_json(artifact / 'label_mapping.json', mapping)
    save_json(artifact / 'config.json', config)
    meta = {'created_utc': datetime.now(timezone.utc).isoformat(), 'environment': environment(),
            'input': input_audit, 'config': config, 'split': split_audit, 'metrics': metrics,
            'validation_feature_count': sum(feature_counts(model).values()),
            'full_feature_count': sum(feature_counts(full).values()),
            'validation_feature_counts': feature_counts(model),
            'full_feature_counts': feature_counts(full),
            'validation_iterations': model.named_steps['classifier'].n_iter_.tolist(),
            'full_iterations': full.named_steps['classifier'].n_iter_.tolist(),
            'timing_seconds': {'validation_fit': fit_seconds, 'full_fit': full_seconds,
                               'train_total': time.perf_counter() - start},
            'artifact_sha256': {p.name: sha256(p) for p in artifact.iterdir() if p.is_file()}}
    save_json(artifact / 'metadata.json', meta)
    print(f'Saved artifacts: {artifact}', flush=True)


def validate_submission(test, submission):
    require(list(submission.columns) == ['id', 'model'], 'Submission columns must be exactly id, model in that order.')
    require(len(submission) == len(test), 'Submission row count differs from test.')
    require(not submission.isna().any().any(), 'Submission contains nulls.')
    for col in ['id', 'model']:
        require(all(isinstance(x, str) for x in submission[col]), f'{col} must contain strings.')
    require(not submission['id'].duplicated().any(), 'Submission IDs are duplicated.')
    require(submission['id'].tolist() == test['id'].tolist(), 'Submission IDs/order differ from test.')
    require(set(submission['model']) <= set(LABELS), 'Submission contains invalid labels.')
    return {'rows': len(submission), 'columns': list(submission), 'valid_labels': True,
            'id_order_equal': True, 'no_nulls': True, 'unique_ids': True,
            'string_values': True, 'read_back_checked': True}


def predict(args):
    start = time.perf_counter()
    output = Path(args.output)
    require(output.suffix == '.parquet', 'Output must use .parquet.')
    require(not output.exists() and not output.with_suffix('.checks.json').exists(), 'Prediction output already exists; choose a new path.')
    require(args.batch_size > 0, 'batch-size must be positive.')
    test, audit = read_input(args.test)
    pipeline, meta = load_artifact(args.artifact_dir, 'pipeline.joblib')
    require(set(pipeline.classes_) == set(LABELS), 'Artifact lacks official classes.')
    mapping = json.loads((Path(args.artifact_dir) / 'label_mapping.json').read_text(encoding='utf-8'))
    require(mapping['model_classes'] == pipeline.classes_.tolist(), 'Saved class mapping is inconsistent.')
    batches = [np.zeros((0, len(pipeline.classes_)))]
    with threadpool_limits(limits=1):
        for offset in range(0, len(test), args.batch_size):
            probabilities = pipeline.predict_proba(test['text'].iloc[offset:offset + args.batch_size])
            require(np.isfinite(probabilities).all() and (probabilities >= 0).all() and
                    (probabilities <= 1).all() and np.allclose(probabilities.sum(axis=1), 1),
                    'Invalid model probabilities.')
            batches.append(probabilities)
        probabilities = np.vstack(batches)
        if args.assignment == 'topic':
            predictions = assign_by_topic(test['text'].tolist(), probabilities, pipeline.classes_).tolist()
        else:
            predictions = pipeline.classes_[probabilities.argmax(axis=1)].tolist()
    result = pd.DataFrame({'id': test['id'].astype('string'), 'model': pd.Series(predictions, dtype='string')})
    # Validate in memory and AGAIN after reading the exported file.
    validate_submission(test, result)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open('xb') as f:
        result.to_parquet(f, index=False, engine='pyarrow')
    checked = validate_submission(test, pd.read_parquet(output))
    checked.update(input=audit, assignment=args.assignment, output_sha256=sha256(output),
                   seconds=time.perf_counter() - start,
                   probability_class_order=pipeline.classes_.tolist(),
                   artifact_model_sha256=meta['artifact_sha256']['pipeline.joblib'])
    save_json(output.with_suffix('.checks.json'), checked)
    print(json.dumps(checked, ensure_ascii=False), flush=True)


def evaluate(args):
    frame, audit = read_input(args.train, labeled=True)
    model, meta = load_artifact(args.artifact_dir, 'validation_pipeline.joblib')
    dummy, _ = load_artifact(args.artifact_dir, 'validation_dummy.joblib')
    require(audit['sha256'] == meta['input']['sha256'], 'Training CSV changed; cannot reuse the recorded holdout.')
    split_path = Path(args.artifact_dir) / 'split.csv'
    require(sha256(split_path) == meta['artifact_sha256']['split.csv'], 'Split manifest changed.')
    split = pd.read_csv(split_path, dtype={'id': 'string'})
    require(split['id'].tolist() == frame['id'].tolist(), 'Split ID order mismatch.')
    directory = Path(args.output_dir)
    require(not directory.exists(), 'Evaluation output directory already exists.')
    directory.mkdir(parents=True)
    metrics = evaluate_models(frame, np.flatnonzero(split['split'].eq('train')),
                              np.flatnonzero(split['split'].eq('validation')), model, dummy, directory)
    require(metrics == meta['metrics'], 'Reloaded validation metrics differ from the training run.')
    print(json.dumps({'metrics': metrics, 'matches_saved_metrics': True}), flush=True)


def cross_validate(args):
    """5-fold grouped CV of the fixed config: argmax vs optional topic assignment. Never reads test."""
    start = time.perf_counter()
    config = json.loads(Path(args.config).read_text(encoding='utf-8'))
    frame, input_audit = read_input(args.train, labeled=True)
    output = Path(args.output_dir)
    require(not output.exists(), 'Use a new output directory; existing runs are never overwritten.')
    groups, _ = connected_groups(frame)
    n_splits = 5
    require(len(set(groups)) >= n_splits, f'Need at least {n_splits} independent groups for CV.')
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=config['seed'])
    folds = []
    for number, (train_idx, valid_idx) in enumerate(splitter.split(frame['text'], frame['model'], groups), 1):
        absent, exact_overlap, normalized_overlap, prompt_overlap = check_disjoint(frame, groups, train_idx, valid_idx)
        train_part, valid = frame.iloc[train_idx], frame.iloc[valid_idx]
        model = build_pipeline(config)
        fit_seconds = fit_checked(model, train_part['text'], train_part['model'])
        with threadpool_limits(limits=1):
            probabilities = model.predict_proba(valid['text'])
        argmax = model.classes_[probabilities.argmax(axis=1)]
        topic = assign_by_topic(valid['text'].tolist(), probabilities, model.classes_)
        report = classification_report(valid['model'], argmax, labels=LABELS, output_dict=True, zero_division=0)
        folds.append({'fold': number, 'train_rows': len(train_idx), 'validation_rows': len(valid_idx),
                      'validation_groups': len(set(groups[valid_idx])), 'absent_classes': absent,
                      'prompt_overlap': prompt_overlap, 'exact_text_overlap': exact_overlap,
                      'normalized_text_overlap': normalized_overlap,
                      'argmax': score(valid['model'], argmax), 'topic': score(valid['model'], topic),
                      'argmax_recall_per_class': {label: report[label]['recall'] for label in LABELS},
                      'fit_seconds': fit_seconds})
        print(json.dumps({k: folds[-1][k] for k in ('fold', 'validation_rows', 'argmax', 'topic')}), flush=True)
    summary = {}
    for mode in ('argmax', 'topic'):
        values = np.array([f[mode]['balanced_accuracy'] for f in folds])
        summary[mode] = {'balanced_accuracy_mean': float(values.mean()), 'balanced_accuracy_std': float(values.std())}
    gains = np.array([f['topic']['balanced_accuracy'] - f['argmax']['balanced_accuracy'] for f in folds])
    summary.update(topic_gain_min=float(gains.min()), topic_gain_mean=float(gains.mean()),
                   sc3_balanced_folds_pass=bool((gains > 0).all() and gains.mean() >= 0.05))
    output.mkdir(parents=True)
    save_json(output / 'cv_metrics.json', {
        'config': config, 'n_splits': n_splits, 'seed': config['seed'], 'input': input_audit,
        'environment': environment(), 'folds': folds, 'summary': summary,
        'seconds': time.perf_counter() - start,
        'note': 'Balanced folds only; unequal per-model counts reduce topic gain (debate 1, E9).'})
    print(json.dumps(summary), flush=True)


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest='command', required=True)
    p = commands.add_parser('train', help='One grouped holdout, then full-data refit; never reads test.')
    p.add_argument('--train', required=True)
    p.add_argument('--config', default='configs/baseline.json')
    p.add_argument('--artifact-dir', required=True)
    p.add_argument('--output-dir', required=True)
    p.set_defaults(func=train)
    p = commands.add_parser('evaluate', help='Reload the held-out model and original split, without refitting.')
    p.add_argument('--train', required=True)
    p.add_argument('--artifact-dir', required=True)
    p.add_argument('--output-dir', required=True)
    p.set_defaults(func=evaluate)
    p = commands.add_parser('predict', help='Offline batched inference using the full-data fitted pipeline.')
    p.add_argument('--test', required=True)
    p.add_argument('--artifact-dir', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--batch-size', type=int, default=256)
    p.add_argument('--assignment', choices=['argmax', 'topic'], default='argmax',
                   help="'topic' = optional per-cluster distinct labels (debate 1); default argmax.")
    p.set_defaults(func=predict)
    p = commands.add_parser('cv', help='5-fold grouped CV: argmax vs topic assignment; never reads test.')
    p.add_argument('--train', required=True)
    p.add_argument('--config', default='configs/baseline.json')
    p.add_argument('--output-dir', required=True)
    p.set_defaults(func=cross_validate)
    p = commands.add_parser('validate', help='Independently read and validate an existing Parquet output.')
    p.add_argument('--test', required=True)
    p.add_argument('--submission', required=True)
    p.set_defaults(func=lambda a: print(json.dumps(validate_submission(read_input(a.test)[0], pd.read_parquet(a.submission)))))
    args = parser.parse_args(argv)
    args.func(args)


if __name__ == '__main__':
    main()
'''
baseline_module = {'__name__': 'embedded_baseline'}
exec(compile(baseline_source, 'src/baseline.py', 'exec'), baseline_module)
for symbol in ('LABELS','normalized','read_input','connected_groups','build_pipeline','require','sha256','environment','validate_submission','main'):
    globals()[symbol] = baseline_module[symbol]


In [ ]:
def available_gib():
    # Kaggle can expose host RAM in /proc even when a cgroup limits the session.
    if os.name == 'nt':
        from ctypes import create_string_buffer, windll
        import struct
        buf = create_string_buffer(64)
        struct.pack_into('I', buf, 0, 64)
        require(windll.kernel32.GlobalMemoryStatusEx(buf), 'Cannot measure available RAM')
        return struct.unpack_from('Q', buf, 16)[0] / 2**30
    import psutil
    available = psutil.virtual_memory().available
    mounts = [(Path('/sys/fs/cgroup'), 'memory.max', 'memory.current'),
              (Path('/sys/fs/cgroup/memory'), 'memory.limit_in_bytes', 'memory.usage_in_bytes')]
    for root, limit_file, usage_file in mounts:
        if (root / limit_file).exists():
            limit = (root / limit_file).read_text().strip()
            if limit != 'max':
                available = min(available, int(limit) - int((root / usage_file).read_text()))
    return max(0, available) / 2**30


def peak_gib():
    if os.name == 'nt':
        import ctypes
        counters = (ctypes.c_size_t * 12)()
        size = 8 + 8 * ctypes.sizeof(ctypes.c_size_t)
        require(ctypes.windll.psapi.GetProcessMemoryInfo(ctypes.c_void_p(-1),
                ctypes.byref(counters), size), 'Cannot read peak working set')
        return ctypes.c_size_t.from_buffer(counters, 8).value / 2**30
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20


def session_seconds_left():
    elapsed = time.monotonic() - START
    if os.name != 'nt':
        import psutil
        elapsed = max(elapsed, time.time() - psutil.Process().create_time())
    return 12 * 3600 - elapsed


def worker_count(free, peak):
    if os.name == 'nt' and free < 6:
        print('Local RAM below 6 GiB: close unused applications; using at most one worker.')
        return 1
    reserve = 2 if os.name == 'nt' else 4
    jobs = min(3, 5, math.floor((free - reserve) / peak))
    return max(0, jobs)


def checkpoint():
    report['elapsed_seconds'] = time.monotonic() - START
    tmp = WORK / 'metrics.json.tmp'
    tmp.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(WORK / 'metrics.json')


In [ ]:
def find_one(filename):
    paths = sorted(p for p in INPUT.rglob(filename) if p.is_file())
    require(len(paths) == 1, f'{filename}: expected one file under {INPUT}, found {len(paths)}: {paths}')
    return paths[0]


names = ['train.csv', 'test.csv', 'ntl_open_pool.parquet', 'folds.csv', 'oof_tfidf.csv',
         'sample_ids_A.csv', 'sample_ids_B.csv']
paths = {name: find_one(name) for name in names}
WORK.mkdir(parents=True, exist_ok=True)
require(not any(WORK.glob('oof_pool_K*.csv')) and not any(WORK.glob('sample_ids_K*.csv'))
        and not any((WORK / f).exists() for f in ('metrics.json', 'submission.parquet', 'folds.csv')),
        'Start with a fresh output directory to avoid mixing separate runs.')
report = {'environment': environment(), 'inputs': {n: {'path': str(p), 'sha256': sha256(p)}
          for n, p in paths.items()}, 'config': CONFIG, 'baseline_oof': BASELINE,
          'variants': {k: {'status': 'pending'} for k in ('K0', 'K1', 'K2', 'K3')}}
print(report['environment'])
checkpoint()
require(report['inputs']['folds.csv']['sha256'] == EXPECTED_FOLDS, 'folds.csv SHA-256 mismatch')
require(report['inputs']['ntl_open_pool.parquet']['sha256'] == EXPECTED_POOL, 'Pool SHA-256 differs from v19')
train, _ = read_input(paths['train.csv'], labeled=True)
test, _ = read_input(paths['test.csv'])
manifest = pd.read_csv(paths['folds.csv'], dtype={'id':'string', 'model':'string', 'prompt_id':'string'})
pd.testing.assert_frame_equal(manifest[['id','prompt_id','model']], train[['id','prompt_id','model']])
folds = manifest.fold
groups, train_keys = connected_groups(train)
require(set(folds) == set(range(1, 6)) and folds.groupby(groups).nunique().max() == 1, 'Invalid real folds')
for fold in range(1, 6):
    require(set(train.model[folds.eq(fold)]) == set(LABELS), f'Incomplete class coverage in fold {fold}')
base_oof = pd.read_csv(paths['oof_tfidf.csv'], dtype={'id':'string'})
require(list(base_oof) == ['id', *LABELS] and base_oof.id.equals(train.id), 'Invalid baseline OOF layout')
baseline_p = base_oof[LABELS].to_numpy()
require(np.isfinite(baseline_p).all() and (baseline_p >= 0).all()
        and np.allclose(baseline_p.sum(axis=1), 1), 'Invalid baseline probabilities')
base_score = balanced_accuracy_score(train.model, base_oof[LABELS].idxmax(axis=1))
require(abs(base_score - BASELINE) < 1e-8, 'Baseline OOF mismatch')
pool = pd.read_parquet(paths['ntl_open_pool.parquet'],
                       columns=['id', 'model', 'text', 'gen_reasoning', 'match_weight']).set_index('id')
require(pool.index.is_unique and not pool.index.hasnans and set(pool.model) == set(LABELS)
        and pool.text.notna().all(), 'Invalid pool IDs, labels or texts')
keys = set(train_keys)
duplicates = sum(normalized(t) in keys for t in pool.text)
report['normalized_text_duplicates'] = duplicates
checkpoint()
require(duplicates == 0, f'{duplicates} pool rows duplicate real train texts')
eligible = pool.loc[pool.model.ne('kimi-k3') | pool.gen_reasoning.eq('high')]
for variant in ('A', 'B'):
    ids = pd.read_csv(paths[f'sample_ids_{variant}.csv'], dtype='string').set_index('id')
    require(ids.index.is_unique and ids.index.isin(pool.index).all(), f'Invalid sample IDs in {variant}')
    require(np.array_equal(ids.model, pool.loc[ids.index, 'model']), f'Sample label mismatch in {variant}')
    require(set(ids.model) == set(LABELS) and ids.model.value_counts().eq(SMALL).all(),
            f'Invalid class counts in {variant}')
    if variant == 'B':
        require(ids.index.isin(eligible.index).all(), 'Ineligible K0 IDs')
        k0_ids = ids.index
import shutil
shutil.copyfile(paths['folds.csv'], WORK / 'folds.csv')
print(f'{len(train)} real rows, {len(pool)} pool rows, {len(eligible)} eligible; {duplicates} duplicate texts')


In [ ]:
def sample_ids(name):
    if name == 'K0':
        return k0_ids
    if name == 'K3':
        return eligible.index
    rng = np.random.default_rng(42)
    size = SMALL if name == 'K1' else LARGE
    cap = float(eligible.match_weight.quantile(.99))
    chosen = []
    for label in LABELS:
        rows = eligible.loc[eligible.model.eq(label)]
        probabilities = None
        if name == 'K2':
            weights = rows.match_weight.clip(upper=cap).to_numpy(dtype=float)
            require(np.isfinite(weights).all() and (weights >= 0).all() and (weights > 0).sum() >= size,
                    f'Invalid or insufficient weights for {label}')
            probabilities = weights / weights.sum()
        pos = rng.choice(len(rows), size=size, replace=False, p=probabilities)
        chosen.extend(rows.index[pos])
    return pd.Index(chosen, name='id')


def fit_predict(real, extra, valid, config, number, prediction_batch=512):
    start = time.monotonic()
    training = pd.concat([real, extra], ignore_index=True)
    with threadpool_limits(1), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        model = build_pipeline(config)
        model.fit(training.text, training.model)
        require(np.any(model.named_steps['classifier'].coef_ != 0), 'Empty fit')
        order = pd.Index(model.classes_).get_indexer(LABELS)
        require((order >= 0).all(), 'Missing probability columns')
        batches = [model.predict_proba(valid.text.iloc[i:i+prediction_batch])[:, order]
                   for i in range(0, len(valid), prediction_batch)]
    p = np.vstack(batches) if batches else np.empty((0, len(LABELS)))
    require(np.isfinite(p).all() and (p >= 0).all() and np.allclose(p.sum(axis=1), 1), 'Invalid probabilities')
    result = pd.DataFrame(p, columns=LABELS, index=valid.id)
    stats = dict(fold=number, seconds=time.monotonic()-start, worker_peak_gib=peak_gib(),
                 real_train=len(real), pool_train=len(extra), predicted_rows=len(valid),
                 convergence_warnings=sum(issubclass(w.category, ConvergenceWarning) for w in caught))
    if number:
        stats['score'] = float(balanced_accuracy_score(valid.model, result.idxmax(axis=1)))
    return result, stats


def run_variant(name):
    info = report['variants'][name]
    start = time.monotonic()
    if name == 'K3':
        scores = [report['variants'][k]['oof'] for k in ('K0','K1','K2')]
        gain = scores[2] - max(scores[:2])
        if gain < MIN_GAIN:
            info.update(status='skipped', reason=f'K2 gain {gain:.8f} below {MIN_GAIN}')
            return
        if session_seconds_left() <= 3600:
            info.update(status='skipped', reason='Session already within final hour')
            return
    ids = sample_ids(name)
    extra = eligible.loc[ids, ['text','model']]
    extra[['model']].to_csv(WORK / f'sample_ids_{name}.csv')
    info.update(status='running', rows=len(extra), counts=extra.model.value_counts().to_dict(), folds=[])
    checkpoint()
    def submit(executor, n):
        mask = folds.eq(n)
        return executor.submit(fit_predict, train.loc[~mask, ['text','model']], extra,
                               train.loc[mask, ['id','text','model']], CONFIG, n)
    free_before = available_gib()
    print(f'{name}: {free_before:.2f} GiB available before pilot', flush=True)
    if os.name != 'nt':
        require(free_before > 4, 'Insufficient RAM for solo pilot with 4 GiB reserve')
    pilot_start = time.monotonic()
    with ProcessPoolExecutor(max_workers=1) as executor:
        first, stats = submit(executor, 1).result()
    pilot_seconds = time.monotonic() - pilot_start
    info['folds'].append(stats)
    free = available_gib()
    jobs = worker_count(free, stats['worker_peak_gib'])
    info.update(pilot_wall_seconds=pilot_seconds, free_gib_after_pilot=free, n_jobs=jobs)
    checkpoint()
    if jobs < 1:
        info.update(status='skipped', reason='No worker fits in measured RAM budget',
                    seconds=time.monotonic()-start)
        if name != 'K3':
            checkpoint()
            raise RuntimeError(f'{name}: {info["reason"]}')
        return
    if name == 'K3':
        budget = session_seconds_left() - 3600
        estimate = pilot_seconds * 4 / jobs
        info.update(estimated_remaining_seconds=estimate, budget_seconds=budget)
        if estimate >= budget:
            info.update(status='skipped', reason='Estimated remaining folds exceed session budget',
                        seconds=time.monotonic()-start)
            return
    parts = [first]
    with ProcessPoolExecutor(max_workers=jobs) as executor:
        futures = [submit(executor, n) for n in range(2, 6)]
        for future in as_completed(futures):
            part, stats = future.result()
            parts.append(part)
            info['folds'].append(stats)
            print(name, stats, flush=True)
            checkpoint()
    oof = pd.concat(parts, verify_integrity=True)
    require(len(oof) == len(train) and train.id.isin(oof.index).all(), 'Incomplete OOF coverage')
    oof = oof.loc[train.id]
    score = float(balanced_accuracy_score(train.model, oof.idxmax(axis=1)))
    oof.to_csv(WORK / f'oof_pool_{name}.csv')
    info['folds'].sort(key=lambda f: f['fold'])
    info.update(status='complete', oof=score, delta=score-BASELINE, seconds=time.monotonic()-start,
                convergence_warnings=sum(f['convergence_warnings'] for f in info['folds']))
    print(name, 'OOF', score, flush=True)
    if name == 'K0' and abs(score - K0_TARGET) > K0_TOLERANCE:
        info.update(status='failed', reason='K0 reproduction outside tolerance')
        for k in ('K1','K2','K3'):
            report['variants'][k].update(status='skipped', reason='K0 reproduction failed')
        checkpoint()
        raise RuntimeError(f'K0 OOF {score:.8f} outside tolerance. Versions: {report["environment"]}')


In [ ]:
for name in ('K0', 'K1', 'K2', 'K3'):
    run_variant(name)
    checkpoint()

complete = [k for k, v in report['variants'].items() if v['status'] == 'complete']
winner = max(complete, key=lambda k: report['variants'][k]['oof'])
report['winner'] = winner
saved = pd.read_csv(WORK / f'sample_ids_{winner}.csv', dtype='string')
with ProcessPoolExecutor(max_workers=1) as executor:
    probs, fit_stats = executor.submit(fit_predict, train[['text','model']],
        eligible.loc[saved.id, ['text','model']], test[['id','text']], CONFIG, 0).result()
submission = probs.idxmax(axis=1).rename('model').reset_index()
validate_submission(test, submission)
submission.to_parquet(WORK / 'submission.parquet', index=False)
main(['validate', '--test', str(paths['test.csv']), '--submission', str(WORK / 'submission.parquet')])
report.update(refit=fit_stats, submission_validated=True, submission_sha256=sha256(WORK / 'submission.parquet'))
checkpoint()
print('Validated submission.parquet. Selected by pooled OOF:', winner)


Pool source: **Name That LLM Open Pool**, version 19, by **crtproduction**. [Dataset](https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool) · [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).